# Team40 FastAPI + ngrok Relay for SageMaker Serverless Endpoint

This notebook sets up a temporary **FastAPI relay** inside SageMaker Studio/JupyterLab and exposes it using **ngrok**.

Flow:

```text
External UI / browser / external server
        ↓ HTTPS ngrok URL
FastAPI relay running in Team40 SageMaker space
        ↓ boto3 using SageMaker execution role
SageMaker endpoint: iti113-team40-heart-disease
```

This avoids putting AWS Access Key / Secret Access Key in an external app. The relay uses the SageMaker execution role attached to the Studio Space.

> Use this only for temporary demo/testing. Stop ngrok and the FastAPI server after testing.


## 1. Check notebook AWS identity

This should show the Team40 SageMaker execution role, for example:

```text
assumed-role/SageMakerExecutionRole-ITI113-Team40/SageMaker
```


In [1]:
import boto3
import json

sts = boto3.client("sts")
identity = sts.get_caller_identity()

print(json.dumps(identity, indent=2))

{
  "UserId": "AROAQUXQWQCIXUD4FMLZL:SageMaker",
  "Account": "044528205969",
  "Arn": "arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team05/SageMaker",
  "ResponseMetadata": {
    "RequestId": "759c2da6-8bd6-4d52-9fb3-c8088d19c9bb",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "759c2da6-8bd6-4d52-9fb3-c8088d19c9bb",
      "x-amz-sts-extended-request-id": "MTphcC1zb3V0aGVhc3QtMTpTOjE3ODcwMjI5NjU3MTI6Ujp5R1g4U2p5Mw==",
      "content-type": "text/xml",
      "content-length": "461",
      "date": "Tue, 18 Aug 2026 03:16:05 GMT"
    },
    "RetryAttempts": 0
  }
}


## 2. Configuration

Set the SageMaker endpoint name and a simple relay API key.

For demo purposes, the relay API key is stored in the notebook variable below. For a more secure setup, store it in AWS Secrets Manager or an environment variable.


In [2]:
REGION = "ap-southeast-1"
ENDPOINT_NAME = "heart-attack-team05-s502-endpoint"

# Change this before sharing the ngrok URL.
# External callers must send this value in the x-api-key header.
RELAY_API_KEY = "team05-demo-key" # change me

print("Region:", REGION)
print("Endpoint:", ENDPOINT_NAME)
print("Relay API key set:", bool(RELAY_API_KEY))

Region: ap-southeast-1
Endpoint: heart-attack-team05-s502-endpoint
Relay API key set: True


## 3. Check that the endpoint exists

This confirms the endpoint is visible from the Team40 notebook role.


In [3]:
import boto3
from botocore.exceptions import ClientError

sm = boto3.client("sagemaker", region_name=REGION)

try:
    endpoint = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint status:", endpoint["EndpointStatus"])
    print("Endpoint ARN:", endpoint["EndpointArn"])
except ClientError as e:
    print("Could not describe endpoint.")
    print(e.response["Error"]["Code"])
    print(e.response["Error"]["Message"])

Endpoint status: InService
Endpoint ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:endpoint/heart-attack-team05-s502-endpoint


## 4. Test direct endpoint invocation from the notebook

Adjust the sample payload if your Team40 endpoint expects a different JSON format.

This example uses common Heart Disease features.


In [4]:
import boto3
import json
from botocore.exceptions import ClientError

runtime = boto3.client("sagemaker-runtime", region_name=REGION)

sample_payload = {
  "State": "Texas",
  "Sex": "Female",
  "GeneralHealth": "Fair",
  "PhysicalHealthDays": 15.0,
  "MentalHealthDays": 30.0,
  "LastCheckupTime": "Within past year (anytime less than 12 months ago)",
  "PhysicalActivities": "Yes",
  "SleepHours": 4.0,
  "RemovedTeeth": "6 or more, but not all",
  "HadAngina": "No",
  "HadStroke": "No",
  "HadAsthma": "No",
  "HadSkinCancer": "No",
  "HadCOPD": "Yes",
  "HadDepressiveDisorder": "Yes",
  "HadKidneyDisease": "No",
  "HadArthritis": "Yes",
  "HadDiabetes": "No",
  "DeafOrHardOfHearing": "No",
  "BlindOrVisionDifficulty": "Yes",
  "DifficultyConcentrating": "No",
  "DifficultyWalking": "Yes",
  "DifficultyDressingBathing": "No",
  "DifficultyErrands": "No",
  "SmokerStatus": "Current smoker - now smokes every day",
  "ECigaretteUsage": "Never used e-cigarettes in my entire life",
  "ChestScan": "No",
  "RaceEthnicityCategory": "White only, Non-Hispanic",
  "AgeCategory": "Age 65 to 69",
  "HeightInMeters": 1.68,
  "WeightInKilograms": 48.53,
  "BMI": 17.27,
  "AlcoholDrinkers": "Yes",
  "HIVTesting": "No",
  "FluVaxLast12": "No",
  "PneumoVaxEver": "Yes",
  "TetanusLast10Tdap": "No, did not receive any tetanus shot in the past 10 years",
  "HighRiskLastYear": "No",
  "CovidPos": "No"
}


try:
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(sample_payload)
    )

    result = response["Body"].read().decode("utf-8")
    print("Raw endpoint response:")
    print(result)

except ClientError as e:
    print("Endpoint invocation failed.")
    print(e.response["Error"]["Code"])
    print(e.response["Error"]["Message"])
except Exception as e:
    print("Endpoint invocation failed.")
    print(type(e).__name__, str(e))

Raw endpoint response:
{"association_score": 0.745162, "threshold": 0.52, "positive_class": true, "classification": "More similar to respondents who reported having had a heart attack", "model_package_version": 2, "disclaimer": "Educational prototype only. This output describes similarity to respondents who reported a previous heart attack. It does not predict a future heart attack and is not a medical diagnosis."}


## 5. Install FastAPI, Uvicorn, and pyngrok

Run this once in the notebook environment.


In [5]:
%pip install -q fastapi uvicorn pyngrok requests

Note: you may need to restart the kernel to use updated packages.


## 6. Create the FastAPI relay app

The relay exposes only:

- `GET /` basic status
- `GET /health` basic health check
- `POST /predict` prediction relay

The external caller must include:

```text
x-api-key: <your relay key>
```

The caller does **not** get AWS credentials and cannot choose the SageMaker endpoint name.


In [6]:
%%writefile relay_api.py

from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
import boto3
import json
import os

REGION = os.environ.get("AWS_REGION", "ap-southeast-1")
ENDPOINT_NAME = os.environ.get("ENDPOINT_NAME", "heart-attack-team05-s502-endpoint")
RELAY_API_KEY = os.environ.get("RELAY_API_KEY", "team05-demo-key")

runtime = boto3.client("sagemaker-runtime", region_name=REGION)

app = FastAPI(title="Team05 SageMaker Endpoint Relay")

# For temporary demo use. For production, restrict allowed_origins.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def home():
    return {
        "status": "running",
        "service": "Team05 FastAPI relay",
        "endpoint": ENDPOINT_NAME,
        "routes": ["GET /health", "POST /predict", "GET /docs"]
    }

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
async def predict(request: Request, x_api_key: str = Header(None)):
    if x_api_key != RELAY_API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")

    try:
        payload = await request.json()
    except Exception:
        raise HTTPException(status_code=400, detail="Request body must be valid JSON")

    # Keep the relay narrow: call only the fixed endpoint configured on the server side.
    try:
        response = runtime.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType="application/json",
            Accept="application/json",
            Body=json.dumps(payload)
        )

        result = response["Body"].read().decode("utf-8")

        try:
            return json.loads(result)
        except Exception:
            return {"raw_result": result}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


Writing relay_api.py


## 7. Start FastAPI server from the notebook

This starts Uvicorn in the background on port `8000`.

If you prefer, you can run the same command in a SageMaker Terminal:

```bash
uvicorn relay_api:app --host 0.0.0.0 --port 8000
```


In [7]:
import os
import subprocess
import time

# Pass config to relay_api.py through environment variables
os.environ["AWS_REGION"] = REGION
os.environ["ENDPOINT_NAME"] = ENDPOINT_NAME
os.environ["RELAY_API_KEY"] = RELAY_API_KEY

# Stop previous server process if this cell was run before
try:
    relay_process.terminate()
    relay_process.wait(timeout=5)
    print("Stopped previous FastAPI server.")
except Exception:
    pass

relay_process = subprocess.Popen(
    ["uvicorn", "relay_api:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(3)
print("FastAPI server process started.")
print("PID:", relay_process.pid)
print("Local URL: http://127.0.0.1:8000")
print("Docs URL:  http://127.0.0.1:8000/docs")

FastAPI server process started.
PID: 1242
Local URL: http://127.0.0.1:8000
Docs URL:  http://127.0.0.1:8000/docs


## 8. Test the local FastAPI relay

This tests the relay locally before exposing it with ngrok.


In [8]:
import requests
import json

local_url = "http://127.0.0.1:8000/predict"

headers = {
    "x-api-key": RELAY_API_KEY,
    "Content-Type": "application/json"
}

response = requests.post(local_url, headers=headers, json=sample_payload, timeout=60)

print("Status code:", response.status_code)
print("Response text:")
print(response.text)

try:
    print("Parsed JSON:")
    print(json.dumps(response.json(), indent=2))
except Exception:
    pass

Status code: 200
Response text:
{"association_score":0.745162,"threshold":0.52,"positive_class":true,"classification":"More similar to respondents who reported having had a heart attack","model_package_version":2,"disclaimer":"Educational prototype only. This output describes similarity to respondents who reported a previous heart attack. It does not predict a future heart attack and is not a medical diagnosis."}
Parsed JSON:
{
  "association_score": 0.745162,
  "threshold": 0.52,
  "positive_class": true,
  "classification": "More similar to respondents who reported having had a heart attack",
  "model_package_version": 2,
  "disclaimer": "Educational prototype only. This output describes similarity to respondents who reported a previous heart attack. It does not predict a future heart attack and is not a medical diagnosis."
}


## 9. Configure ngrok

You need a free ngrok account and auth token.

1. Go to ngrok dashboard.
2. Copy your auth token.
3. Paste it below.

Do **not** commit the token to GitHub.


In [10]:
from getpass import getpass
from pyngrok import ngrok

NGROK_AUTH_TOKEN = getpass("")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("ngrok auth token configured for this session.")

ngrok auth token configured for this session.


## 10. Start ngrok tunnel

This creates a temporary public HTTPS URL forwarding to local port `8000`.


In [11]:
from pyngrok import ngrok

# Close old tunnels from this notebook session
try:
    for tunnel in ngrok.get_tunnels():
        ngrok.disconnect(tunnel.public_url)
except Exception:
    pass

public_tunnel = ngrok.connect(8000, "http")
public_url = public_tunnel.public_url

print("Public ngrok URL:")
print(public_url)
print("FastAPI docs:")
print(public_url + "/docs")
print("Prediction route:")
print(public_url + "/predict")

Public ngrok URL:
https://hardened-triceps-handbook.ngrok-free.dev
FastAPI docs:
https://hardened-triceps-handbook.ngrok-free.dev/docs
Prediction route:
https://hardened-triceps-handbook.ngrok-free.dev/predict


### if using public ngrok URL: add /predidct to end 
eg. https://massager-unnatural-device.ngrok-free.dev/predict

## 11. Test ngrok public URL from the notebook

This simulates an external client calling the public ngrok URL.


In [12]:
import requests
import json

headers = {
    "x-api-key": RELAY_API_KEY,
    "Content-Type": "application/json"
}

response = requests.post(public_url + "/predict", headers=headers, json=sample_payload, timeout=60)

print("Status code:", response.status_code)
print("Response text:")
print(response.text)

try:
    print("Parsed JSON:")
    print(json.dumps(response.json(), indent=2))
except Exception:
    pass

Status code: 200
Response text:
{"association_score":0.745162,"threshold":0.52,"positive_class":true,"classification":"More similar to respondents who reported having had a heart attack","model_package_version":2,"disclaimer":"Educational prototype only. This output describes similarity to respondents who reported a previous heart attack. It does not predict a future heart attack and is not a medical diagnosis."}
Parsed JSON:
{
  "association_score": 0.745162,
  "threshold": 0.52,
  "positive_class": true,
  "classification": "More similar to respondents who reported having had a heart attack",
  "model_package_version": 2,
  "disclaimer": "Educational prototype only. This output describes similarity to respondents who reported a previous heart attack. It does not predict a future heart attack and is not a medical diagnosis."
}


## 12. Example external caller code

Give this pattern to the external UI/server developer.

They only need:

- ngrok URL
- relay API key
- JSON payload format

They do **not** need AWS access keys.


In [13]:
import json

example_code = """import requests

url = "{url}/predict"
headers = {{
    "x-api-key": "{api_key}",
    "Content-Type": "application/json"
}}

payload = {payload}

response = requests.post(url, headers=headers, json=payload, timeout=60)
print(response.status_code)
print(response.text)
""".format(
    url=public_url,
    api_key=RELAY_API_KEY,
    payload=json.dumps(sample_payload, indent=4)
)

print(example_code)

import requests

url = "https://hardened-triceps-handbook.ngrok-free.dev/predict"
headers = {
    "x-api-key": "team05-demo-key",
    "Content-Type": "application/json"
}

payload = {
    "State": "Texas",
    "Sex": "Female",
    "GeneralHealth": "Fair",
    "PhysicalHealthDays": 15.0,
    "MentalHealthDays": 30.0,
    "LastCheckupTime": "Within past year (anytime less than 12 months ago)",
    "PhysicalActivities": "Yes",
    "SleepHours": 4.0,
    "RemovedTeeth": "6 or more, but not all",
    "HadAngina": "No",
    "HadStroke": "No",
    "HadAsthma": "No",
    "HadSkinCancer": "No",
    "HadCOPD": "Yes",
    "HadDepressiveDisorder": "Yes",
    "HadKidneyDisease": "No",
    "HadArthritis": "Yes",
    "HadDiabetes": "No",
    "DeafOrHardOfHearing": "No",
    "BlindOrVisionDifficulty": "Yes",
    "DifficultyConcentrating": "No",
    "DifficultyWalking": "Yes",
    "DifficultyDressingBathing": "No",
    "DifficultyErrands": "No",
    "SmokerStatus": "Current smoker - now smokes every da

## 13. Stop ngrok and FastAPI after demo

Run this when testing is complete.

This helps prevent unexpected endpoint invocations and costs.


In [14]:
# Stop ngrok tunnels
try:
    ngrok.kill()
    print("ngrok tunnels stopped.")
except Exception as e:
    print("ngrok stop issue:", e)

# Stop FastAPI server
try:
    relay_process.terminate()
    relay_process.wait(timeout=5)
    print("FastAPI server stopped.")
except Exception as e:
    print("FastAPI stop issue:", e)

ngrok tunnels stopped.


FastAPI server stopped.


## Troubleshooting

### `AccessDeniedException` from SageMaker Runtime

The Team40 execution role needs permission to invoke the endpoint:

```json
{
  "Effect": "Allow",
  "Action": "sagemaker:InvokeEndpoint",
  "Resource": "arn:aws:sagemaker:ap-southeast-1:044528205969:endpoint/iti113-team40-heart-disease"
}
```

### `404` or endpoint not found

Check that the endpoint name is exactly:

```text
iti113-team40-heart-disease
```

### External caller gets `401 Invalid API key`

Make sure they send the header:

```text
x-api-key: <RELAY_API_KEY>
```

### ngrok URL changes

Free ngrok URLs usually change every time you restart the tunnel.

### Security note

This relay should expose only fixed routes such as `/predict`. Do not create routes that accept arbitrary Python code, arbitrary AWS actions, S3 paths, or arbitrary SageMaker endpoint names.
